In [ ]:
import numpy as np
from typing import List, Callable
from scipy.optimize import rosen, approx_fprime
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from math import pi, cos, exp, sqrt

: 

In [ ]:
Vector = List[float]

def grad(f):
    return approx_fprime(f)

def f2(x: Vector): return (x[1]**2 + x[0]**2 - 1)**2 + (x[0] + x[1] - 1)**2

def rosenbroke(x: Vector): return (1-x[0])**2 + 100*(x[1] - x[0]**2)**2

def rostrigin(x: Vector): return 20 + (x[0]**2 - 10*cos(2*pi*x[0])) + (x[1]**2 - 10*cos(2*pi*x[1]))

def eckley(x: Vector): return -20 * exp(-0.2 * sqrt(0.5 * (x[0]**2 + x[1]**2))) - exp(0.5*(cos(2*pi*x[0]) + cos(2*pi*x[1]))) + exp(1) + 20

def himmelbau(x: Vector): return (x[0]**2 + x[1] - 11)**2 + (x[0] + x[1]**2 - 7)**2

In [ ]:
def Adam(f: Callable, x0: Vector, alpha = 0.08, eps_1 = 1e-8, eps_2 = 1e-6,fd_step = 1e-6, max_iter = 1000, gamma_v = 0.95, gamma_s = 0.9999):
    x = np.array(x0, dtype = float)
    n = x.size
    mv = np.zeros(n)
    ms = np.zeros(n)

    s, v = ms, mv

    grad = lambda point: approx_fprime(point, f, fd_step)
    g = grad(x)
    

    traj = [x.copy()]
    k = 0

    while np.linalg.norm(g) > eps_2 and k<max_iter:
        k +=1

        v = gamma_v * v + (1 - gamma_v) * g
        s = gamma_s * s + (1 - gamma_s) * (g * g)

        v_hat = v / (1 - gamma_v ** k)
        s_hat = s / (1 - gamma_s ** k)

        x = x - alpha * v_hat / (np.sqrt(s_hat) + eps_1)
        g = grad(x)
        traj.append(x.copy())
    
    return x, f(x), k, np.array(traj)
        

In [ ]:
def nesterov(f: Callable, x0: Vector, eps=1e-6, max_iter=1000, alpha=1e-4, beta=0.9, fd_step=1e-6):
    x = np.atleast_1d(np.asarray(x0, dtype=float))

    grad = lambda v: approx_fprime(v, f, fd_step)

    x_prev = x.copy()
    g = grad(x)
    traj = [x.copy()]
    k = 0

    while np.linalg.norm(g) > eps and k < max_iter:
        k += 1

        y = x + beta * (x - x_prev)

        g_y = grad(y)
        x_new = y - alpha * g_y

        x_prev = x
        x = x_new
        g = grad(x)

        traj.append(x.copy())

    return x, f(x), k, np.array(traj)

In [ ]:
def quadratic_interpolation(x_1: Vector, function:Callable,
                            delta_x=1e-2,
                            epsilon_1=1e-6,
                            epsilon_2=1e-6,
                            max_iter=50,
                            x_lower=0.0,
                            x_upper=None):

    # ---- 0. приводим x_1 в допустимый интервал ----
    x_1 = max(x_1, x_lower)
    if x_upper is not None:
        x_1 = min(x_1, x_upper)

    x_2 = x_1 + delta_x
    if x_upper is not None:
        x_2 = min(x_2, x_upper)

    f1 = function(x_1)
    f2 = function(x_2)

    if f1 > f2:
        x_3 = x_1 + 2.0 * delta_x
        if x_upper is not None:
            x_3 = min(x_3, x_upper)
    else:
        x_3 = x_1 - delta_x
        if x_3 < x_lower:
            x_3 = x_1 + 2.0 * delta_x
            if x_upper is not None:
                x_3 = min(x_3, x_upper)

    f3 = function(x_3)

    if x_2 == x_3:
        x_3 = x_1 + 3.0 * delta_x
        if x_upper is not None:
            x_3 = min(x_3, x_upper)
        f3 = function(x_3)

    for _ in range(max_iter):
        denom = ((x_2 - x_3) * f1
                 + (x_3 - x_1) * f2
                 + (x_1 - x_2) * f3)

        if abs(denom) < 1e-16:
            # точки почти на одной прямой — парабола не определена
            break

        num = 0.5 * ((x_2**2 - x_3**2) * f1
                     + (x_3**2 - x_1**2) * f2
                     + (x_1**2 - x_2**2) * f3)
        x_min = num / denom

        # ---- 3. ограничения на x_min ----
        if not np.isfinite(x_min):
            break
        if x_min < x_lower:
            x_min = 0.5 * (x_1 + x_lower) if x_1 > x_lower \
                    else x_1 + 0.5 * delta_x
        if x_upper is not None and x_min > x_upper:
            x_min = 0.5 * (x_1 + x_upper)

        f_min = function(x_min)

        # NaN/Inf — выходим, вернём лучшую из имеющихся
        if not np.isfinite(f_min):
            break

        # ---- 4. критерий сходимости ----
        dx_ok = abs(x_min - x_1) < epsilon_2 * max(abs(x_min), 1e-12)
        df_ok = abs(f_min - f1) < epsilon_1 * max(abs(f_min), 1e-12)
        if dx_ok and df_ok:
            return x_min, f_min

        # ---- 5. новый триплет из 4 точек ----
        pts = [(x_1, f1), (x_2, f2), (x_3, f3), (x_min, f_min)]
        pts.sort(key=lambda p: p[0])           # по x

        # индекс минимума по f
        idx = min(range(4), key=lambda i: pts[i][1])

        if idx == 0:
            triple = pts[0:3]
        elif idx == 3:
            triple = pts[1:4]
        else:
            if idx == 2 or pts[idx - 1][1] < pts[idx + 1][1]:
                triple = pts[idx - 1:idx + 2]
            else:
                triple = pts[idx:idx + 3]

        (x_1, f1), (x_2, f2), (x_3, f3) = triple

        if x_1 < x_lower:
            x_1 = x_lower
            f1 = function(x_1)
        if x_upper is not None and x_3 > x_upper:
            x_3 = x_upper
            f3 = function(x_3)

        if x_2 >= x_3:
            x_2 = 0.5 * (x_1 + x_3)
            f2 = function(x_2)

    candidates = [(x_1, f1), (x_2, f2), (x_3, f3)]
    return min(candidates, key=lambda p: p[1])

In [ ]:
def dfp(f, x0, df=None, eps=1e-4, max_iter=100, fd_step=1e-6):
    x = np.array(x0, dtype=float)
    n = x.size
    H = np.eye(n)
    k = 0
    traj = [x.copy()]

    grad = df if df is not None else (lambda v: approx_fprime(v, f, fd_step))
    g = grad(x)

    while np.linalg.norm(g) > eps and k < max_iter:
        d = -H @ g
        if g @ d >= 0:
            d = -g
            H = np.eye(n)

        dphi0 = g @ d
        fun1d = lambda alpha: f(x + alpha * d)

        if dphi0 < 0:
            alpha_guess = min(1.0, 1.0 / max(np.linalg.norm(g), 1e-12))
            alpha, _ = quadratic_interpolation(
                alpha_guess, fun1d,
                delta_x=max(alpha_guess * 1e-2, 1e-4),
                epsilon_1=1e-6, epsilon_2=1e-6,
                x_lower=0.0,
                x_upper=2.0 / max(np.linalg.norm(g), 1e-12)
            )
        else:
            alpha = 1e-6

        # Armijo в цикле
        phi0 = fun1d(0.0)
        c1 = 1e-4
        n_back = 0
        while fun1d(alpha) > phi0 + c1 * alpha * dphi0 and n_back < 30:
            alpha *= 0.5
            n_back += 1
        if alpha <= 1e-10:
            alpha = 1e-10

        x_new = x + alpha * d
        g_new = grad(x_new)

        s = x_new - x
        y = g_new - g
        sy = s @ y

        if sy > 1e-9:
            Hy = H @ y
            yHy = y @ Hy
            if yHy > 1e-9:
                H = H + np.outer(s, s) / sy - np.outer(Hy, Hy) / yHy
            else:
                H = np.eye(n)
        else:
            H = np.eye(n)

        x, g = x_new, g_new
        k += 1
        traj.append(x.copy())

    return x, f(x), k, np.array(traj)

In [ ]:
def coord_descent(f, x0, eps=1e-4, max_iter=1000, fd_step=1e-8):
    x = np.array(x0, dtype=float)
    n = x.size
    k = 0
    traj = [x.copy()]

    grad = lambda v: approx_fprime(v, f, fd_step)

    while k < max_iter:
        x_old = x.copy()

        for i in range(n):
            g = grad(x)
            if abs(g[i]) < 1e-14:
                continue

            e = np.zeros(n)
            e[i] = 1.0

            # направление спуска по i-й координате
            d = -g[i] * e
            dphi0 = g @ d                      # = -g[i]**2 < 0

            fun1d = lambda alpha: f(x + alpha * d)

            if dphi0 < 0:
                alpha_guess = min(1.0, 1.0 / max(abs(g[i]), 1e-12))
                alpha, _ = quadratic_interpolation(
                    alpha_guess, fun1d,
                    delta_x=max(alpha_guess * 1e-2, 1e-4),
                    epsilon_1=1e-6, epsilon_2=1e-6,
                    x_lower=0.0,
                    x_upper=2.0 / max(abs(g[i]), 1e-12),
                )
            else:
                alpha = 1e-6

            # бэктрекинг по Армихо
            phi0 = fun1d(0.0)
            c1 = 1e-4
            n_back = 0
            while fun1d(alpha) > phi0 + c1 * alpha * dphi0 and n_back < 30:
                alpha *= 0.5
                n_back += 1
            if alpha <= 1e-10:
                alpha = 1e-10

            x = x + alpha * d
            traj.append(x.copy())

        k += 1
        if np.linalg.norm(x - x_old) < eps:
            break

    return x, f(x), k, np.array(traj)

In [ ]:
def plot_trajectory(f, traj, title, exact=None,
                    pad=0.25, n_grid=100):
    """
    Строит линии уровня f и накладывает траекторию.
    Пределы осей и уровни изолиний подбираются автоматически:
      - xlim / ylim считаются по траектории (+ exact) с запасом pad
      - уровни изолиний — логарифмическая сетка по значениям f в окне
    """
    # --- 1. Авто-пределы по траектории ---
    all_pts = traj.copy()
    if exact is not None:
        all_pts = np.vstack([all_pts, np.array(exact)])

    x_min, x_max = all_pts[:, 0].min(), all_pts[:, 0].max()
    y_min, y_max = all_pts[:, 1].min(), all_pts[:, 1].max()

    # защита от вырожденного диапазона (все точки в одной координате)
    if x_max - x_min < 1e-6:
        x_min -= 1.0; x_max += 1.0
    if y_max - y_min < 1e-6:
        y_min -= 1.0; y_max += 1.0

    dx = (x_max - x_min) * pad
    dy = (y_max - y_min) * pad
    xlim = (x_min - dx, x_max + dx)
    ylim = (y_min - dy, y_max + dy)

    # --- 2. Сетка для линий уровня ---
    xs = np.linspace(xlim[0], xlim[1], n_grid)
    ys = np.linspace(ylim[0], ylim[1], n_grid)
    X, Y = np.meshgrid(xs, ys)
    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i, j] = f(np.array([X[i, j], Y[i, j]]))

    # --- 3. Авто-уровни: логарифмическая сетка ---
    zmin = Z.min()
    zmax = Z.max()
    if zmax - zmin < 1e-12:
        levels = np.array([zmin])
    else:
        levels = np.logspace(np.log10(zmin + 1e-6),
                             np.log10(zmax), 30)

    # --- 4. Рисуем ---
    fig, ax = plt.subplots(figsize=(8, 7))
    cs = ax.contour(X, Y, Z, levels=levels,
                    cmap='viridis', linewidths=0.8)
    ax.clabel(cs, inline=True, fontsize=7, fmt='%.1e')

    ax.plot(traj[:, 0], traj[:, 1], 'r.-',
            linewidth=1.6, markersize=9, label='траектория')
    ax.plot(traj[0, 0], traj[0, 1], 'go',
            markersize=13, label='старт')
    ax.plot(traj[-1, 0], traj[-1, 1], 'b*',
            markersize=18, label='финиш')

    for i, (xi, yi) in enumerate(traj):
        ax.annotate(str(i), (xi, yi),
                    textcoords="offset points",
                    xytext=(7, 5), fontsize=8, color='darkred')

    if exact is not None:
        ax.plot(exact[0], exact[1], 'kX', markersize=14,
                label='точный минимум')

    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_title(title, fontsize=12)
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.grid(True, linestyle=':', alpha=0.4)
    ax.legend(loc='best', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_trajectory_3d_plotly(f, traj, title, exact=None,
                              pad=0.25, n_grid=100,
                              xlim=None, ylim=None, zlim=None,
                              colorscale='RdBu_r',
                              surface_opacity=0.9):
    """
    Интерактивная 3D-поверхность + траектория (plotly).
    Вращение — ЛКМ, зум — колесо, панорама — Shift+ЛКМ,
    сброс — двойной клик.
    """
    # --- 1. Пределы по осям X, Y ---
    all_pts = traj.copy()
    if exact is not None:
        all_pts = np.vstack([all_pts, np.array(exact)])

    if xlim is None:
        x_min, x_max = all_pts[:, 0].min(), all_pts[:, 0].max()
        if x_max - x_min < 1e-6:
            x_min -= 1.0; x_max += 1.0
        dx = (x_max - x_min) * pad
        xlim = (x_min - 2*dx, x_max + 2*dx)

    if ylim is None:
        y_min, y_max = all_pts[:, 1].min(), all_pts[:, 1].max()
        if y_max - y_min < 1e-6:
            y_min -= 1.0; y_max += 1.0
        dy = (y_max - y_min) * pad
        ylim = (y_min - 2*dy, y_max + 2*dy)

    # --- 2. Сетка поверхности ---
    xs = np.linspace(xlim[0], xlim[1], n_grid)
    ys = np.linspace(ylim[0], ylim[1], n_grid)
    X, Y = np.meshgrid(xs, ys)
    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i, j] = f(np.array([X[i, j], Y[i, j]]))

    # --- 3. Z-диапазон для вида ---
    Ztraj = np.array([f(p) for p in traj])
    zmin = min(Z.min(), Ztraj.min())
    zmax_raw = max(Z.max(), Ztraj.max())
    zmax_cap = np.percentile(Z, 92)
    zmax = min(zmax_raw, zmax_cap)
    if zlim is not None:
        zmin, zmax = zlim

    # --- 4. Поверхность ---
    fig = go.Figure()

    fig.add_surface(
        x=X, y=Y, z=Z,
        colorscale=colorscale,
        opacity=surface_opacity,
        showscale=True,
        colorbar=dict(title='f(x)', len=0.6),
        contours={
            "z": {"show": True,
                  "usecolormap": False,
                  "highlightcolor": "black",
                  "project": {"z": True}}
        },
        name='поверхность'
    )

    # --- 5. Траектория: белая обводка + красная линия ---
    fig.add_scatter3d(
        x=traj[:, 0], y=traj[:, 1], z=Ztraj,
        mode='lines',
        line=dict(color='white', width=8),
        showlegend=False, hoverinfo='skip'
    )
    fig.add_scatter3d(
        x=traj[:, 0], y=traj[:, 1], z=Ztraj,
        mode='lines+markers+text',
        text=[str(i) for i in range(len(traj))],
        textposition='top center',
        textfont=dict(color='black', size=10),
        line=dict(color='red', width=4),
        marker=dict(size=5, color='red',
                    line=dict(color='white', width=1)),
        name='траектория'
    )

    # --- 6. Вертикальные нити ---
    for i, (xi, yi, zi) in enumerate(zip(traj[:, 0], traj[:, 1], Ztraj)):
        fig.add_scatter3d(
            x=[xi, xi], y=[yi, yi], z=[zmin, zi],
            mode='lines',
            line=dict(color='gray', width=1, dash='dot'),
            showlegend=False, hoverinfo='skip'
        )

    # --- 7. Старт / финиш / точный минимум ---
    fig.add_scatter3d(
        x=[traj[0, 0]], y=[traj[0, 1]], z=[Ztraj[0]],
        mode='markers',
        marker=dict(size=10, color='lime',
                    line=dict(color='black', width=2)),
        name='старт'
    )
    fig.add_scatter3d(
        x=[traj[-1, 0]], y=[traj[-1, 1]], z=[Ztraj[-1]],
        mode='markers',
        marker=dict(size=14, color='blue', symbol='diamond',
                    line=dict(color='white', width=2)),
        name='финиш'
    )
    if exact is not None:
        fig.add_scatter3d(
            x=[exact[0]], y=[exact[1]], z=[f(np.array(exact))],
            mode='markers',
            marker=dict(size=12, color='black', symbol='x'),
            name='точный минимум'
        )

    # --- 8. Оформление ---
    fig.update_layout(
        title=title,
        width=950, height=750,
        scene=dict(
            xaxis=dict(title='x1', range=list(xlim)),
            yaxis=dict(title='x2', range=list(ylim)),
            zaxis=dict(title='f(x)', range=[zmin, zmax]),
            aspectmode='cube',
            camera=dict(eye=dict(x=1.6, y=-1.6, z=1.0))
        ),
        legend=dict(x=0.02, y=0.98,
                    bgcolor='rgba(255,255,255,0.7)')
    )

    fig.show()

In [ ]:
%timeit Adam(rosen, [0.5, 8.7])

In [ ]:
x_adam, f_adam, k_adam, traj_adam = Adam(rosen, [1, 3.5])
print(f"Итог: x* = {x_adam}, f* = {f_adam:.3e}, k = {k_adam}\n")

In [ ]:
plot_trajectory(rosen, traj_adam, title='Функция Розенброка, метод Adam', exact=(1, 1))

In [ ]:
plot_trajectory_3d_plotly(rosen, traj_adam, title='Функция Розенброка, метод Adam', exact=(1, 1))

In [ ]:
%timeit Adam(rostrigin, [0.5, 4.7])

In [ ]:
x_adam, f_adam, k_adam, traj_adam = Adam(rostrigin, [1, 0])
print(f"Итог: x* = {x_adam}, f* = {f_adam:.3e}, k = {k_adam}\n")

In [ ]:
plot_trajectory(rostrigin, traj_adam, title = 'метод Adam, функция Ростригина', exact = (0, 0))

In [ ]:
plot_trajectory_3d_plotly(rostrigin, traj_adam, title='метод Adam, функция Ростригина')

In [ ]:
%timeit Adam(eckley, [0.5, 4.7])

In [ ]:
x_adam, f_adam, k_adam, traj_adam = Adam(eckley, [0.5, 4.7])
print(f"Итог: x* = {x_adam}, f* = {f_adam:.3e}, k = {k_adam}\n")

In [ ]:
plot_trajectory(eckley, traj_adam, title = 'Функция Экли, метод Adam')

In [ ]:
plot_trajectory_3d_plotly(eckley, traj_adam, title = 'Функция Экли, метод Adam')

In [ ]:
%timeit nesterov(rosen, [0.2, 0.4])

In [ ]:
x_nesterov, f_nesterov, k_nesterov, traj_nesterov = nesterov(rosen, [0.2, 0.4])
print(f"Итог: x* = {x_nesterov}, f* = {f_nesterov:.3e}, k = {k_nesterov}\n")

In [ ]:
plot_trajectory(rosen, traj_nesterov, title = 'функция Розенброка, метод Нестерова')

In [ ]:
plot_trajectory_3d_plotly(rosen, traj_nesterov, title = 'функция Розенброка, метод Нестерова')

In [ ]:
%timeit nesterov(rostrigin, [0.5, 4.7])

In [ ]:
x_nesterov, f_nesterov, k_nesterov, traj_nesterov = nesterov(rostrigin, [0.4, 0])
print(f"Итог: x* = {x_nesterov}, f* = {f_nesterov:.3e}, k = {k_nesterov}\n")

In [ ]:
plot_trajectory(rostrigin, traj_nesterov, title = 'функция Ростригина, метод Нестерова', exact=(0, 0))

In [ ]:
plot_trajectory_3d_plotly(rostrigin, traj_nesterov, title = 'функция Ростригина, метод Нестерова')

In [ ]:
%timeit nesterov(eckley, [5, 4.7])

In [ ]:
x_nesterov, f_nesterov, k_nesterov, traj_nesterov = nesterov(eckley, [0, 0.7], alpha=1e-2)
print(f"Итог: x* = {x_nesterov}, f* = {f_nesterov:.3e}, k = {k_nesterov}\n")

In [ ]:
plot_trajectory(eckley, traj_nesterov, title = 'функция Экли, метод Нестерова', exact=(0, 0))

In [ ]:
plot_trajectory_3d_plotly(eckley, traj_nesterov, title = 'функция Экли, метод Нестерова', exact=(0, 0))

In [ ]:
%timeit dfp(rosenbroke, [0.5, 8.7])
x_dfp, f_dfp, k_dfp, traj_dfp = dfp(rosenbroke, [0.5, 8.7])
print(f"Итог: x* = {x_dfp}, f* = {f_dfp:.3e}, k = {k_dfp}\n")

In [ ]:
plot_trajectory(rosenbroke, traj_dfp,title = 'Функция Розенброка', exact=(1,1))

In [ ]:
plot_trajectory_3d_plotly(rosenbroke, traj_dfp, title='Функция Розенброка', exact=(1, 1))

In [ ]:
%timeit dfp(rostrigin, [0.2, 0.4])
x_dfp, f_dfp, k_dfp, traj_dfp = dfp(rostrigin, [0.2, 0.4])
print(f"Итог: x* = {x_dfp}, f* = {f_dfp:.3e}, k = {k_dfp}\n")

In [ ]:
plot_trajectory(rostrigin, traj_dfp, title='Функция Ростригина', exact=(0, 0))

In [ ]:
plot_trajectory_3d_plotly(rostrigin, traj_dfp, 'Функция Ростригина', exact = (0, 0))

In [ ]:
%timeit dfp(eckley, [0.2, 0.4])
x_dfp, f_dfp, k_dfp, traj_dfp = dfp(eckley, [0.2, 0.4])
print(f"Итог: x* = {x_dfp}, f* = {f_dfp:.3e}, k = {k_dfp}\n")

In [ ]:
plot_trajectory(eckley, traj_dfp, title='Функция Экли', exact=(0, 0))

In [ ]:
plot_trajectory_3d_plotly(eckley, traj_dfp, 'Функция Экли', exact = (0, 0))

In [ ]:
%timeit dfp(himmelbau, [0.5, 8.7])
x_dfp, f_dfp, k_dfp, traj_dfp = dfp(himmelbau, [1, -5])
print(f"Итог: x* = {x_dfp}, f* = {f_dfp:.3e}, k = {k_dfp}\n")

In [ ]:
plot_trajectory(himmelbau, traj_dfp, title='Функция Химмельбау', exact=(3.5844, -1.8481))

In [ ]:
plot_trajectory_3d_plotly(himmelbau, traj_dfp, 'Функция Химельбау', exact = (3.5844, -1.8481))

In [ ]:
%timeit coord_descent(rosen, [3, 1])

In [ ]:
x_coord, f_coord, k_coord, traj_coord = coord_descent(rosen, [3, 1])
print(f"Итог: x* = {x_coord}, f* = {f_coord:.3e}, k = {k_coord}\n")

In [ ]:
plot_trajectory(rosen, traj_coord, "Покоординатный спускб, функция Розенброка", exact = (1, 1))

In [ ]:
plot_trajectory_3d_plotly(rosen, traj_coord, "Покоординатный спуск, функция Розенброка", exact = (1, 1))

In [ ]:
%timeit coord_descent(rostrigin, [1, 1])

In [ ]:
x_coord, f_coord, k_coord, traj_coord = coord_descent(rostrigin, [1, 1])
print(f"Итог: x* = {x_coord}, f* = {f_coord:.3e}, k = {k_coord}\n")

In [ ]:
plot_trajectory(rostrigin, traj_coord, "Покоординатный спускб, функция Ростригина", exact = (0, 0))

In [ ]:
plot_trajectory_3d_plotly(rostrigin, traj_coord, "Покоординатный спускб, функция Розенброка", exact = (1, 1))

In [ ]:
%timeit coord_descent(eckley, [2, 3])

In [ ]:
x_coord, f_coord, k_coord, traj_coord = coord_descent(eckley, [1, 1])
print(f"Итог: x* = {x_coord}, f* = {f_coord:.3e}, k = {k_coord}\n")

In [ ]:
plot_trajectory(eckley, traj_coord, "Покоординатный спускб, функция Ростригина", exact = (0, 0))

In [ ]:
plot_trajectory_3d_plotly(eckley, traj_coord, "Покоординатный спускб, функция Ростригина", exact = (0, 0))

In [ ]:
%timeit coord_descent(himmelbau, [1, 1])

In [ ]:
x_coord, f_coord, k_coord, traj_coord = coord_descent(himmelbau, [1, 1])
print(f"Итог: x* = {x_coord}, f* = {f_coord:.3e}, k = {k_coord}\n")

In [ ]:
plot_trajectory(himmelbau, traj_coord, "Покоординатный спускб, функция Химмельбау", exact = (3, 2))

In [ ]:
plot_trajectory_3d_plotly(himmelbau, traj_coord, "Покоординатный спускб, функция Химмельбау", exact = (3, 2))